# KORE Colab 빠른 시작

아래 코드 셀 하나로 설치, 테스트, 계약 검증, 샘플 실험을 끝낸다. 샘플 데이터는 배관 점검용이라 연구 결과로 쓰지 않는다.

- 무료 Colab은 연결이 끊길 수 있다. `USE_DRIVE = True`이면 결과가 내 Google Drive에 남고, 같은 명령을 다시 실행하면 끝난 부분은 건너뛴다.
- 저장소가 비공개이면 비밀 토큰을 이 노트북에 붙여 넣지 않는다. Colab 왼쪽의 Secrets(열쇠 아이콘)에 저장해서 쓴다.

In [ ]:
# KORE 빠른 시작: 설치, 테스트, 샘플 실행을 한 셀에서 끝낸다
import os, subprocess, sys

REPO_URL = 'https://github.com/OWNER/kore.git'  # 팀 저장소 주소로 바꾼다
BRANCH = 'main'
WORKDIR = '/content/kore'
USE_DRIVE = True  # True이면 결과를 내 Google Drive에 저장한다

if 'OWNER' in REPO_URL:
    raise SystemExit('REPO_URL을 팀 저장소 주소로 바꾼 뒤 다시 실행한다.')

def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, text=True, capture_output=True)
    print(r.stdout[-2500:] + r.stderr[-1200:])
    if r.returncode != 0:
        raise RuntimeError('명령 실패: ' + cmd)

if os.path.isdir(WORKDIR):
    sh('git fetch origin && git checkout ' + BRANCH + ' && git pull --ff-only', cwd=WORKDIR)
else:
    sh('git clone --branch ' + BRANCH + ' ' + REPO_URL + ' ' + WORKDIR)

sh(sys.executable + ' -m pip install -q -e ".[dev]"', cwd=WORKDIR)
sh(sys.executable + ' -m pytest -q', cwd=WORKDIR)
sh(sys.executable + ' -m kore validate --knowledge data/sample_knowledge.csv --questions data/sample_eval.csv', cwd=WORKDIR)

OUT_ROOT = '/content/results'
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        OUT_ROOT = '/content/drive/MyDrive/kore_results'
    except ImportError:
        print('Colab이 아니라서 Drive를 쓰지 않는다.')
OUT_DIR = OUT_ROOT + '/sample_run'
sh(sys.executable + ' -m kore run --config configs/default.json --knowledge data/sample_knowledge.csv --questions data/sample_eval.csv --out ' + OUT_DIR, cwd=WORKDIR)
print('결과 폴더:', OUT_DIR)

In [ ]:
# 결과 표 확인(조건×분할별 평균과 95% 구간)
import pandas as pd
pd.read_csv(OUT_DIR + '/results.csv')